# Import Statements

In [54]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import train_test_split
from collections import Counter
from nltk.tokenize import word_tokenize
import nltk
import json
import torch.optim as optim
from sklearn.metrics import f1_score
import wandb

In [55]:
# wandb.login()

# Set Device

In [56]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Train Dataset

In [57]:
df_train = pd.read_csv('../../data/train_lemmastop.csv')
df_train['processed_text'] = df_train['processed_text'].fillna('')
df_train.head()

,id,text,anger,fear,joy,sadness,surprise,emotions,clean_text,processed_text
0,0,the dentist that did the work apparently did a...,1,0,0,1,0,['anger' 'sadness'],the dentist that did the work apparently did a...,dentist work apparently lousy job year teeth d...
1,1,i'm gonna absolutely ~~suck~~ be terrible duri...,0,1,0,1,0,['fear' 'sadness'],i am going to absolutely be terrible during my...,going absolutely terrible first sexual experience
2,2,"bridge: so leave me drowning calling houston, ...",0,1,0,1,0,['fear' 'sadness'],bridge so leave me drowning calling houston an...,bridge leave drowning calling houston let lung...
3,3,after that mess i went to see my now ex-girlfr...,1,1,0,1,0,['anger' 'fear' 'sadness'],after that mess i went to see my now exgirlfri...,mess went see exgirlfriend school refused driv...
4,4,"as he stumbled i ran off, afraid it might some...",0,1,0,0,0,['fear'],as he stumbled i ran off afraid it might someh...,stumbled ran afraid might somehow affect job s...


In [58]:
emotion_cols = ['anger', 'fear', 'joy', 'sadness', 'surprise']
labels = df_train[emotion_cols].values

In [59]:
all_texts = ' '.join(df_train['processed_text'])
words = word_tokenize(all_texts)
word_counts = Counter(words)

# Vocabulary

In [60]:
# Create the vocabulary
# Start with our special tokens
vocab = {'<PAD>': 0, '<UNK>': 1}
for word, count in word_counts.items():
    if count > 1:
        vocab[word] = len(vocab)
print(f"Vocabulary size: {len(vocab)} unique words")

Vocabulary size: 4873 unique words


In [61]:
with open('../../models/lstm/LSTM_Model_att_dls_vocab.json', 'w') as f:
    json.dump(vocab, f)

# Tokenization

In [62]:
def text_to_sequence(text, vocab):
    """Converts a text string to a sequence of integers using the vocab."""
    tokens = word_tokenize(text)
    return [vocab.get(word, 1) for word in tokens]

In [63]:
sequences = [text_to_sequence(text, vocab) for text in df_train['processed_text']]
seq_lengths = [len(s) for s in sequences]
MAX_LEN = int(np.percentile(seq_lengths, 95))
print(f"Using a max sequence length of: {MAX_LEN}")

Using a max sequence length of: 18


In [64]:
def pad_sequence(seq, max_len):
    """Pads a sequence to max_len. Truncates if longer."""
    if len(seq) > max_len:
        return seq[:max_len]  # Truncate
    else:
        return seq + [0] * (max_len - len(seq))

In [65]:
padded_sequences = [pad_sequence(s, MAX_LEN) for s in sequences]

X = torch.tensor(padded_sequences, dtype=torch.long)
y = torch.tensor(labels, dtype=torch.float32)
print(f"Shape of features (X): {X.shape}")
print(f"Shape of labels (y): {y.shape}")

Shape of features (X): torch.Size([6827, 18])
Shape of labels (y): torch.Size([6827, 5])


# Emotion Dataset Class

In [66]:
class EmotionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = features
        self.labels = labels

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

In [67]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
train_dataset = EmotionDataset(X_train, y_train)
val_dataset = EmotionDataset(X_val, y_val)

# Data Loaders

In [68]:
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

print("DataLoaders created successfully.")
print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")

DataLoaders created successfully.
Train batches: 171
Validation batches: 43


# LSTM

In [69]:
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=False, 
            dropout=dropout,
            batch_first=True
        )
        
        self.fc = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text shape: [batch_size, seq_len]
        embedded = self.embedding(text)
        outputs, (hidden, cell) = self.lstm(embedded)
        
        hidden = hidden[-1,:,:]

        hidden = self.dropout(hidden)
        prediction = self.fc(hidden)
       
        return prediction

# Attention-based LSTM

In [70]:
class LSTMAttentionModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, n_layers, dropout):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        
        self.lstm = nn.LSTM(
            embedding_dim, 
            hidden_dim, 
            num_layers=n_layers, 
            bidirectional=False, 
            dropout=dropout, 
            batch_first=True
        )
        
        # The Attention Layer
        self.attention = nn.Linear(hidden_dim, 1)
        
        # The Final Classification Layer
        self.fc = nn.Linear(hidden_dim, output_dim)
        
        self.dropout = nn.Dropout(dropout)

    def forward(self, text):
        # text: [batch_size, seq_len]
        embedded = self.embedding(text)
        
        # output: [batch_size, seq_len, hidden_dim * 2]
        output, (hidden, cell) = self.lstm(embedded)
        
        # ATTENTION MECHANISM
        # energy: [batch_size, seq_len, 1]
        energy = torch.tanh(self.attention(output))
        
        # weights: [batch_size, seq_len, 1]
        weights = F.softmax(energy, dim=1)
        
        # weighted: [batch_size, seq_len, hidden_dim * 2]
        weighted = output * weights
        
        # context_vector: [batch_size, hidden_dim * 2]
        context_vector = torch.sum(weighted, dim=1)
        
        # Pass the Context Vector to the classifier
        final_input = self.dropout(context_vector)
        prediction = self.fc(final_input)
        
        return prediction

# Parameters

In [71]:
VOCAB_SIZE = len(vocab)
EMBEDDING_DIM = 100 
HIDDEN_DIM = 128     
OUTPUT_DIM = 5      
N_LAYERS = 4         
DROPOUT = 0.4        
LEARNING_RATE = 1e-3
N_EPOCHS = 50

# Initialize WandB

In [72]:
wandb.init(
    project="24f1002325-t32025", 
    entity="24f1002325-iit-madras",
    name=f"LSTM-Run-dls-Attention-layers{N_LAYERS}-lr{LEARNING_RATE}-epochs{N_EPOCHS}",
    config={
        "model_type": "LSTM",
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "n_layers": N_LAYERS,
        "dropout": DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": N_EPOCHS,
        "batch_size": BATCH_SIZE
    }
)

# Initialize Model

In [73]:
model = LSTMAttentionModel(
    VOCAB_SIZE, 
    EMBEDDING_DIM, 
    HIDDEN_DIM, 
    OUTPUT_DIM, 
    N_LAYERS, 
    DROPOUT
)
model = model.to(device)  

In [74]:
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

# F1 Score

In [75]:
def calculate_f1(preds, y_true, threshold=0.5):
    y_pred = torch.sigmoid(preds)
    y_pred = (y_pred > threshold).int()
    
    y_pred_cpu = y_pred.cpu().numpy()
    y_true_cpu = y_true.cpu().numpy()
    
    return f1_score(y_true_cpu, y_pred_cpu, average='macro', zero_division=0)

In [76]:
def train_epoch(model, iterator, optimizer, criterion):
    model.train() 
    epoch_loss = 0
    
    for features, labels in iterator:
        features = features.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()

        predictions = model(features)

        loss = criterion(predictions, labels)

        loss.backward()

        optimizer.step()
        
        epoch_loss += loss.item()
        
    return epoch_loss / len(iterator)

In [77]:
def evaluate_epoch(model, iterator, criterion):
    model.eval()  
    epoch_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for features, labels in iterator:
            features = features.to(device)
            labels = labels.to(device)
            
            predictions = model(features)
            loss = criterion(predictions, labels)
            
            epoch_loss += loss.item()
            
            all_preds.append(predictions)
            all_targets.append(labels)

    all_preds = torch.cat(all_preds, dim=0)
    all_targets = torch.cat(all_targets, dim=0)

    f1 = calculate_f1(all_preds, all_targets)
    
    return epoch_loss / len(iterator), f1

# Model Training 

In [78]:
print("Starting training.")

for epoch in range(N_EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_f1 = evaluate_epoch(model, val_loader, criterion)

    if (epoch + 1) % 5 == 0:
        print(f'Epoch: {epoch+1:02}')
        print(f'\tTrain Loss: {train_loss:.3f}')
        print(f'\t Val. Loss: {val_loss:.3f} |  Val. Macro F1: {val_f1:.4f}')

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_f1
    })

wandb.finish()
print("Training finished.")

Starting training.
Epoch: 05
	Train Loss: 0.469
	 Val. Loss: 0.501 |  Val. Macro F1: 0.4181
Epoch: 10
	Train Loss: 0.286
	 Val. Loss: 0.487 |  Val. Macro F1: 0.6050
Epoch: 15
	Train Loss: 0.179
	 Val. Loss: 0.514 |  Val. Macro F1: 0.6623
Epoch: 20
	Train Loss: 0.113
	 Val. Loss: 0.570 |  Val. Macro F1: 0.6812
Epoch: 25
	Train Loss: 0.075
	 Val. Loss: 0.664 |  Val. Macro F1: 0.6897
Epoch: 30
	Train Loss: 0.053
	 Val. Loss: 0.714 |  Val. Macro F1: 0.6904
Epoch: 35
	Train Loss: 0.038
	 Val. Loss: 0.776 |  Val. Macro F1: 0.6907
Epoch: 40
	Train Loss: 0.040
	 Val. Loss: 0.772 |  Val. Macro F1: 0.7067
Epoch: 45
	Train Loss: 0.031
	 Val. Loss: 0.832 |  Val. Macro F1: 0.7064
Epoch: 50
	Train Loss: 0.030
	 Val. Loss: 0.888 |  Val. Macro F1: 0.7040


epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_loss,██▇▇▇▅▅▄▄▃▃▃▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_loss,▂▂▂▂▁▁▁▁▁▁▂▂▂▂▂▃▃▄▃▄▄▅▄▅▄▅▆▆▆▇▆▆▇▆▇▇██▇█
val_macro_f1,▁▁▂▃▆▆▇▇▇▇▇▇█▇████▇█████████████████████
epoch,50
train_loss,0.03033
val_loss,0.88821
val_macro_f1,0.70402


Training finished.


# Save Model

In [79]:
torch.save(model.state_dict(), '../../models/lstm/lstm_att_dls_model.pth')